# Proyecto - Clasificación Competitiva de Textos

Este notebook deja de depender de Google Drive y usa un pipeline más competitivo para la competencia:

- `StratifiedKFold` para validación más estable
- modelo preentrenado fuerte para español
- chunking con `stride` para textos largos
- ensemble de folds para la submission final

La idea no es solo entrenar un modelo, sino producir una estimación de validación más confiable y una predicción de test más robusta.

In [ ]:
# Instala solo si hace falta. Si ya tienes el entorno listo, puedes saltarte esta celda.
%pip install -q -U "transformers>=4.49.0" "datasets>=3.2.0" "accelerate>=1.2.1" "sentencepiece" "scikit-learn" "torch" "ipywidgets"


In [ ]:
import gc
import json
import os
import random
import re
import warnings
from typing import Any, cast
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision("high")

print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
TRAIN_PATH = "train.csv"
EVAL_PATH = "eval.csv"

TEXT_COL = "text"
TARGET_COL = "decade"
ID_COL = "id"

# Modelo recomendado para español. Alternativas razonables:
# - "BSC-LT/MrBERT"
# - "FacebookAI/xlm-roberta-base"
# - "dccuchile/bert-base-spanish-wwm-cased"
MODEL_NAME = "BSC-LT/MrBERT-es"

# Configuración ajustada para una RTX 4090 de 24 GB.
MAX_LENGTH = 384
STRIDE = 96
N_SPLITS = 3
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 1
WARMUP_RATIO = 0.1
DATALOADER_NUM_WORKERS = 10
SEED = 42

OUTPUT_DIR = "competition_runs"
SUBMISSION_PATH = "submission_competitive.csv"
OOF_PATH = "oof_predictions.csv"
TEST_PROBS_PATH = "test_probabilities.npy"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
BF16 = torch.cuda.is_available()
FP16 = False

print("Modelo:", MODEL_NAME)
print("Device:", device)
print("bf16:", BF16)


In [ ]:
def clean_text(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = text.replace("\ufeff", " ").replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


train_df = pd.read_csv(TRAIN_PATH)
eval_df = pd.read_csv(EVAL_PATH)

train_df[TEXT_COL] = train_df[TEXT_COL].map(clean_text)
eval_df[TEXT_COL] = eval_df[TEXT_COL].map(clean_text)

train_df["doc_id"] = np.arange(len(train_df))
eval_df["doc_id"] = eval_df[ID_COL].astype(int)

label_encoder = LabelEncoder()
train_df["label_id"] = label_encoder.fit_transform(train_df[TARGET_COL])

num_labels = len(label_encoder.classes_)
print("train shape:", train_df.shape)
print("eval shape:", eval_df.shape)
print("número de clases:", num_labels)
print("clases:", label_encoder.classes_[:5], "...")

char_lens = train_df[TEXT_COL].str.len()
word_lens = train_df[TEXT_COL].str.split().str.len()
print("\nCuantiles de longitud en caracteres:")
print(char_lens.quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_string())
print("\nCuantiles de longitud en palabras:")
print(word_lens.quantile([0.5, 0.75, 0.9, 0.95, 0.99]).to_string())


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8 if (BF16 or FP16) else None)


def build_chunk_dataset(df: pd.DataFrame, with_labels: bool = True) -> Dataset:
    texts = df[TEXT_COL].tolist()
    doc_ids = df["doc_id"].tolist()

    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    chunk_doc_ids = [doc_ids[idx] for idx in sample_mapping]

    features = {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "doc_id": chunk_doc_ids,
    }

    if "token_type_ids" in tokenized:
        features["token_type_ids"] = tokenized["token_type_ids"]

    if with_labels:
        labels = df["label_id"].tolist()
        features["labels"] = [labels[idx] for idx in sample_mapping]

    return Dataset.from_dict(features)


def aggregate_predictions(doc_ids, probs, method: str = "mean"):
    prob_cols = [f"p_{i}" for i in range(probs.shape[1])]
    tmp = pd.DataFrame({"doc_id": doc_ids})
    prob_df = pd.DataFrame(probs, columns=prob_cols)
    tmp = pd.concat([tmp, prob_df], axis=1)

    if method == "max":
        agg = tmp.groupby("doc_id", sort=False)[prob_cols].max().reset_index()
    else:
        agg = tmp.groupby("doc_id", sort=False)[prob_cols].mean().reset_index()

    pred_ids = agg[prob_cols].to_numpy().argmax(axis=1)
    return agg["doc_id"].to_numpy(), pred_ids, agg[prob_cols].to_numpy()


def compute_chunk_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}


def predict_dataset(trainer: Trainer, dataset: Dataset):
    # cast evita falsos positivos de tipado de Pylance con datasets.Dataset
    return trainer.predict(cast(Any, dataset))


In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_probs = np.zeros((len(train_df), num_labels), dtype=np.float32)
test_probs_folds = []
fold_scores = []

eval_chunk_ds = build_chunk_dataset(eval_df, with_labels=False)
eval_chunk_doc_ids = np.array(eval_chunk_ds["doc_id"])
eval_ds_for_trainer = eval_chunk_ds.remove_columns(["doc_id"])

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label_id"]), start=1):
    print(f"\n========== Fold {fold}/{N_SPLITS} ==========")

    fold_train = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val = train_df.iloc[val_idx].reset_index(drop=True)

    train_ds = build_chunk_dataset(fold_train, with_labels=True)
    val_ds = build_chunk_dataset(fold_val, with_labels=True)

    train_ds_for_trainer = train_ds.remove_columns(["doc_id"])
    val_ds_for_trainer = val_ds.remove_columns(["doc_id"])

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=num_labels,
    )
    model.config.use_cache = False

    fold_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
    os.makedirs(fold_dir, exist_ok=True)
    args = TrainingArguments(
        output_dir=fold_dir,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        num_train_epochs=EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        optim="adamw_torch_fused",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True,
        bf16=BF16,
        fp16=FP16,
        tf32=True,
        gradient_checkpointing=False,
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        dataloader_pin_memory=True,
        report_to="none",
        seed=SEED + fold,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds_for_trainer,
        eval_dataset=val_ds_for_trainer,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_chunk_metrics,
    )

    trainer.train()

    val_pred = predict_dataset(trainer, val_ds_for_trainer)
    val_probs_chunks = torch.softmax(torch.tensor(val_pred.predictions), dim=-1).numpy()
    val_doc_ids = np.array(val_ds["doc_id"])
    val_doc_ids_out, val_pred_ids, val_probs_docs = aggregate_predictions(val_doc_ids, val_probs_chunks, method="mean")

    val_truth = (
        fold_val[["doc_id", "label_id"]]
        .drop_duplicates("doc_id")
        .set_index("doc_id")
        .loc[val_doc_ids_out]["label_id"]
        .to_numpy()
    )
    fold_acc = accuracy_score(val_truth, val_pred_ids)
    fold_scores.append(fold_acc)
    oof_probs[val_doc_ids_out] = val_probs_docs
    print(f"Fold {fold} doc-level accuracy: {fold_acc:.5f}")

    test_pred = predict_dataset(trainer, eval_ds_for_trainer)
    test_probs_chunks = torch.softmax(torch.tensor(test_pred.predictions), dim=-1).numpy()
    _, _, test_probs_docs = aggregate_predictions(eval_chunk_doc_ids, test_probs_chunks, method="mean")
    test_probs_folds.append(test_probs_docs)

    del model, trainer, train_ds, val_ds, train_ds_for_trainer, val_ds_for_trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nFold scores:", [round(x, 5) for x in fold_scores])
print("CV mean accuracy:", round(float(np.mean(fold_scores)), 5))


In [ ]:
oof_pred_ids = oof_probs.argmax(axis=1)
oof_acc = accuracy_score(train_df["label_id"], oof_pred_ids)
print("OOF accuracy global:", round(float(oof_acc), 5))

oof_df = train_df[["doc_id", TARGET_COL, "label_id"]].copy()
oof_df["pred_label_id"] = oof_pred_ids
oof_df["pred_decade"] = label_encoder.inverse_transform(oof_pred_ids)
oof_df.to_csv(OOF_PATH, index=False)
print("OOF guardado en:", OOF_PATH)


In [ ]:
test_probs = np.mean(test_probs_folds, axis=0)
test_pred_ids = test_probs.argmax(axis=1)
test_pred_labels = label_encoder.inverse_transform(test_pred_ids)

submission = eval_df[[ID_COL]].copy()
submission[TARGET_COL] = test_pred_labels
submission.to_csv(SUBMISSION_PATH, index=False)
np.save(TEST_PROBS_PATH, test_probs)

print("Submission guardada en:", SUBMISSION_PATH)
print("Probabilidades de test guardadas en:", TEST_PROBS_PATH)
submission.head()


## Cómo empujarlo más arriba

Si quieres subir más en leaderboard, las siguientes mejoras suelen rendir más que retocar una sola tasa de aprendizaje:

1. Probar 3 a 4 modelos distintos y hacer ensemble de probabilidades.
2. Correr varias seeds y promediar.
3. Subir `MAX_LENGTH` a `384` o `512` si la GPU lo soporta.
4. Probar `N_SPLITS = 5` para un ensemble más estable.
5. Revisar manualmente errores frecuentes por década y por longitud.
6. Si el reglamento lo permite, hacer pseudo-labeling con ejemplos de `eval.csv`.

Orden práctico:

- primero: `BSC-LT/MrBERT-es`
- segundo: `FacebookAI/xlm-roberta-base`
- tercero: `dccuchile/bert-base-spanish-wwm-cased`

Si una sola GPU no alcanza, baja `TRAIN_BATCH_SIZE` y sube `GRAD_ACCUM_STEPS`.